<a href="https://colab.research.google.com/github/encoras/Introduction-to-OpenCV/blob/master/amber_img_cut.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

tar -xzvf archyvas.tar.gz

In [4]:
# ========================== 1. INSTALIACIJA IR IMPORTAI ==========================
!pip install opencv-python-headless scikit-learn pandas matplotlib seaborn -q

from tqdm.auto import tqdm
import cv2
import numpy as np
import pandas as pd
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab.patches import cv2_imshow

# ========================== 2. KONFIGŪRACIJA ==========================
DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/kevir_12_mazi"   # ← pakeisk į savo kelią

# HSV ribos gintarui (labai gerai veikia ant šviesaus fono)
#LOWER_AMBER = np.array([ 6,  20,  25])   # H, S, V
#UPPER_AMBER = np.array([ 40, 205, 180])

#LOWER_AMBER = np.array([ 10,  70,  30 ])   # H min ≈12-2, S min ≈73-3, V min ≈37-7
#UPPER_AMBER = np.array([ 45, 255, 255 ])   # H max ≈37+8, S max, V max
LOWER_AMBER = np.array([  0,  30,  15 ])
UPPER_AMBER = np.array([ 50, 255, 255 ])

# Morfologija
KERNEL = np.ones((5,5), np.uint8)

def segment_dark_amber(hsv):
    # Tamsiems gintarams – platesnis H, mažesnis V minimumas
    lower_dark = np.array([ 0,  70,  1 ])
    upper_dark = np.array([ 90, 255, 180 ])  # V iki 180, kad neperimtų šviesaus fono
    return cv2.inRange(hsv, lower_dark, upper_dark)

def segment_light_amber(hsv):
    lower_light = np.array([ 8,  70,  80 ])
    upper_light = np.array([ 63, 255, 255 ])
    return cv2.inRange(hsv, lower_light, upper_light)



# ========================== 3. SEGMENTAVIMO FUNKCIJA ==========================
def segment_amber(image_path):
    img = cv2.imread(str(image_path))

    if img is None:
        return None, None



    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    #mask = cv2.inRange(hsv, LOWER_AMBER, UPPER_AMBER)
    # Sujungimas
    mask_dark  = segment_dark_amber(hsv)
    mask_light = segment_light_amber(hsv)
    mask = cv2.bitwise_or(mask_dark, mask_light)


    # Valymas
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, KERNEL)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, KERNEL)

    # Imame didžiausią kontūrą (kad nebūtų triukšmo)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None

    largest = max(contours, key=cv2.contourArea)
    mask_clean = np.zeros(mask.shape, np.uint8)
    cv2.drawContours(mask_clean, [largest], -1, 255, -1)

    return img, mask_clean


In [5]:
def crop_center_224(img, cx, cy, target_size=224):
    """
    Iškerpa kvadratą target_size×target_size su centru ties (cx, cy)
    Jei trūksta pikselių – užpildo juodu fonu
    """
    half = target_size // 2

    # Norimos ribos
    x_start = cx - half
    y_start = cy - half
    x_end   = cx + half
    y_end   = cy + half

    # Tikriname, ar telpa į originalų paveikslėlį
    h, w = img.shape[:2]

    # Jei visiškai telpa – paprastas crop
    if x_start >= 0 and y_start >= 0 and x_end <= w and y_end <= h:
        return img[y_start:y_end, x_start:x_end]

    # Jei netelpa – kuriame didesnį juodą kvadratą ir įdedame turimą dalį
    result = np.zeros((target_size, target_size, 3), dtype=np.uint8)

    # kiek realiai įdedame
    src_x_start = max(0, x_start)
    src_y_start = max(0, y_start)
    src_x_end   = min(w, x_end)
    src_y_end   = min(h, y_end)

    dst_x_start = max(0, -x_start)
    dst_y_start = max(0, -y_start)
    dst_x_end   = dst_x_start + (src_x_end - src_x_start)
    dst_y_end   = dst_y_start + (src_y_end - src_y_start)

    result[dst_y_start:dst_y_end, dst_x_start:dst_x_end] = \
        img[src_y_start:src_y_end, src_x_start:src_x_end]

    return result

In [6]:
def crop_and_save_224(image_path, output_path):
    img, mask = segment_amber(image_path)
    if img is None or mask is None:
        return False

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return False

    largest = max(contours, key=cv2.contourArea)

    # Gauname centroidą
    M = cv2.moments(largest)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
    else:
        # atsarginis variantas
        x, y, w, h = cv2.boundingRect(largest)
        cx = x + w // 2
        cy = y + h // 2

    # Kerpame tiksliai pagal centrą
    cropped_224 = crop_center_224(img, cx, cy, target_size=224)

    # Išsaugome
    cv2.imwrite(str(output_path), cropped_224)
    return True

In [ ]:

# ================================================
# KONFIGŪRACIJA
# ================================================

DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/kevir_12_mazi"   # originalus kelias
NEW_DATASET_PATH = "/content/amber_cropped_224"                        # nauja direktorija

# Kiek padding'o pridėti aplink gintarą (rekomenduojama 10-20)
PADDING = 20

# Sukuriame naują struktūrą
Path(NEW_DATASET_PATH).mkdir(parents=True, exist_ok=True)

# ================================================
# Pagrindinė funkcija: iškirpti gintarą ir išsaugoti 224x224
# ================================================





# ================================================
# Vykdymas: einame per visas klases ir nuotraukas
# ================================================

print("🔄 Kuriama nauja 224x224 duomenų bazė su iškirptais gintarais...\n")

for nk_folder in sorted(Path(DATASET_PATH).glob("NK*")):
    class_name = nk_folder.name
    new_class_dir = Path(NEW_DATASET_PATH) / class_name
    new_class_dir.mkdir(exist_ok=True)

    image_list = list(nk_folder.glob("*.*"))
    saved_count = 0

    print(f"Apdorojama klasė: {class_name} ({len(image_list)} nuotraukų)")

    for img_path in tqdm(image_list):
        if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']:
            continue

        new_filename = img_path.stem + "_" + str(np.random.randint(10000,50000)) + img_path.suffix
        output_path = new_class_dir / new_filename

        success = crop_and_save_224(img_path, output_path)
        if success:
            saved_count += 1

    print(f"   → Išsaugota {saved_count} nuotraukų\n")

print("✅ Baigta! Nauja duomenų bazė sukurta:")
print(f"   {NEW_DATASET_PATH}")